# Warm-Start Official Failure Handoff v1

Purpose: convert the latest warm-start official replay failure into a training-objective handoff. This notebook does not train or promote a model. It screens poor 63-day windows, feature deltas, gate gaps, and the next objective family using repo-owned helpers.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison: `sequence_trend_joint_warm_start_official_eval_v1`.
- Role: diagnose / route next training objective.
- Changed layer: `model_selection` and `training_objective_handoff`; no runtime promotion.
- Gate: target relative return >= +30pp and MaxDD <= 15%.
- External references: Qlib (`https://github.com/microsoft/qlib`, `https://arxiv.org/abs/2009.11189`), FinRL ensemble/risk-control precedent (`https://github.com/AI4Finance-Foundation/FinRL`), DAgger imitation-learning caution (`https://proceedings.mlr.press/v15/ross11a.html`), and PBO / purged validation discipline (`https://doi.org/10.21314/JCF.2016.322`).

Historical reflection:
- Closest success: warm-start official helper now provides a reliable daily-path gate source.
- Closest failure: `sequence_trend_joint_warm_start_official_eval_v1` confirmed no full-gate candidate.
- Not a duplicate: this notebook diagnoses bad windows and feature-state differences to choose the next objective instead of changing target thresholds or rerunning PPO.

In [1]:

from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.oos_diagnostics import (
    aggregate_feature_differences,
    aggregate_poor_windows,
    compute_feature_differences,
    find_poor_windows,
)
from src.soxl_quant.app.training_objective_handoff import (
    derive_training_objective_handoff,
    render_training_objective_handoff,
)
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXP = "warm_start_official_failure_handoff_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXP
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_EXP = "sequence_trend_joint_warm_start_official_eval_v1"
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / SOURCE_EXP
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
WINDOW_SIZE = 63
TOP_WINDOWS = 10
TOP_FEATURES = 30
WALK_FORWARD_WINDOWS = {
    "wf_2022": {"train": ("2010-03-11", "2020-12-31"), "valid": ("2021-01-01", "2021-12-31"), "test": ("2022-01-01", "2022-12-31")},
    "wf_2023_2024": {"train": ("2010-03-11", "2022-12-31"), "valid": ("2023-01-01", "2023-12-31"), "test": ("2024-01-01", "2024-12-31")},
}
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://proceedings.mlr.press/v15/ross11a.html",
    "https://doi.org/10.21314/JCF.2016.322",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.floating):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def feature_group(feature: str) -> str:
    name = str(feature)
    if name.startswith(("soxx_", "relative_", "tracking_", "spread_", "beta_")) or "soxx" in name.lower():
        return "soxx_relative"
    if any(token in name for token in ("vol", "vix", "atr", "range")):
        return "volatility_stress"
    if any(token in name for token in ("drawdown", "dd", "drop", "cum_drop")):
        return "drawdown_path"
    if any(token in name for token in ("anchor", "trend", "slope", "ma", "momentum", "return")):
        return "trend_momentum"
    if any(token in name for token in ("rsi", "stoch", "reversal", "gap")):
        return "reversal_reentry"
    return "other"

source_selection = json.loads((SOURCE_CARD / "selection.json").read_text(encoding="utf-8"))
source_summary = json.loads((SOURCE_CARD / "summary.json").read_text(encoding="utf-8"))
warm_metrics = pd.read_csv(SOURCE_CARD / "warm_start_official_metrics.csv")
candidate_summary = pd.read_csv(SOURCE_CARD / "candidate_summary.csv")
profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
print("source decision", source_selection.get("decision"))
print("selected", source_selection.get("selected_candidate"))
print("rows", len(warm_metrics))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[49610:MainThread](2026-05-26 16:56:03,825) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[49610:MainThread](2026-05-26 16:56:04,450) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[49610:MainThread](2026-05-26 16:56:04,451) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


source decision warm_start_official_eval_no_full_gate_no_promotion
selected trend_plain_contract
rows 6


In [2]:

# Build feature frames by fold, using the same profile packs as the diagnostic actor.
feature_cache: dict[str, pd.DataFrame] = {}
for fold, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    features = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").sort_index()
    feature_cache[fold] = features

feature_cache_summary = pd.DataFrame([
    {
        "fold": fold,
        "feature_start": str(frame.index.min().date()),
        "feature_end": str(frame.index.max().date()),
        "features": int(frame.shape[1]),
    }
    for fold, frame in feature_cache.items()
])
feature_cache_summary.to_csv(CARD / "feature_cache_summary.csv", index=False)
display(feature_cache_summary)


[49610:MainThread](2026-05-26 16:56:06,391) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[49610:MainThread](2026-05-26 16:56:06,393) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[49610:MainThread](2026-05-26 16:56:06,397) INFO - qlib.timer - [log.py:127] - Time cost: 0.008s | Init data Done


[49610:MainThread](2026-05-26 16:56:08,836) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[49610:MainThread](2026-05-26 16:56:08,837) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[49610:MainThread](2026-05-26 16:56:08,838) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


,fold,feature_start,feature_end,features
0,wf_2022,2011-03-11,2022-12-30,59
1,wf_2023_2024,2011-03-11,2024-12-31,59


In [3]:

DIAG_DIR = CARD / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)
windows_by_label: dict[str, pd.DataFrame] = {}
features_by_label: dict[str, pd.DataFrame] = {}
diagnostic_manifest = []
for row in warm_metrics.to_dict("records"):
    candidate = str(row["candidate"])
    fold = str(row["fold"])
    label = f"{candidate}__{fold}"
    daily_path = ROOT / str(row["daily_path"])
    daily = pd.read_csv(daily_path, parse_dates=["datetime"])
    windows = find_poor_windows(
        daily,
        window_size=WINDOW_SIZE,
        top_n=TOP_WINDOWS,
        target_excess=TARGET_RELATIVE_RETURN,
        target_max_dd=TARGET_MAX_DD,
    )
    feature_frame = feature_cache[fold].loc[daily["datetime"].min():daily["datetime"].max()]
    feature_diffs = compute_feature_differences(feature_frame, windows, daily=daily, top_n=TOP_FEATURES)
    out_dir = DIAG_DIR / label
    out_dir.mkdir(parents=True, exist_ok=True)
    windows.to_csv(out_dir / "poor_windows.csv", index=False)
    feature_diffs.to_csv(out_dir / "feature_differences.csv", index=False)
    windows_by_label[label] = windows
    features_by_label[label] = feature_diffs
    diagnostic_manifest.append({
        "label": label,
        "candidate": candidate,
        "fold": fold,
        "daily_path": str(daily_path.relative_to(ROOT)),
        "poor_windows": int(len(windows)),
        "feature_differences": int(len(feature_diffs)),
        "diagnostic_dir": str(out_dir.relative_to(ROOT)),
    })

manifest_df = pd.DataFrame(diagnostic_manifest).sort_values(["candidate", "fold"]).reset_index(drop=True)
manifest_df.to_csv(CARD / "diagnostic_manifest.csv", index=False)
display(manifest_df)


,label,candidate,fold,daily_path,poor_windows,feature_differences,diagnostic_dir
0,base_plain_contract__wf_2022,base_plain_contract,wf_2022,runtime/model_selection/card/sequence_trend_jo...,10,30,runtime/model_selection/card/warm_start_offici...
1,base_plain_contract__wf_2023_2024,base_plain_contract,wf_2023_2024,runtime/model_selection/card/sequence_trend_jo...,10,30,runtime/model_selection/card/warm_start_offici...
2,trend_plain_contract__wf_2022,trend_plain_contract,wf_2022,runtime/model_selection/card/sequence_trend_jo...,10,30,runtime/model_selection/card/warm_start_offici...
3,trend_plain_contract__wf_2023_2024,trend_plain_contract,wf_2023_2024,runtime/model_selection/card/sequence_trend_jo...,10,30,runtime/model_selection/card/warm_start_offici...
4,trend_plus_slope_low_cap15_weighted__wf_2022,trend_plus_slope_low_cap15_weighted,wf_2022,runtime/model_selection/card/sequence_trend_jo...,10,30,runtime/model_selection/card/warm_start_offici...
5,trend_plus_slope_low_cap15_weighted__wf_2023_2024,trend_plus_slope_low_cap15_weighted,wf_2023_2024,runtime/model_selection/card/sequence_trend_jo...,10,30,runtime/model_selection/card/warm_start_offici...


In [4]:

clusters = aggregate_poor_windows(
    windows_by_label,
    target_excess=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
)
feature_clusters = aggregate_feature_differences(features_by_label, top_n=50)
if not feature_clusters.empty:
    feature_clusters["feature_group"] = feature_clusters["feature"].map(feature_group)
    group_summary = (
        feature_clusters.assign(abs_mean_delta_z=feature_clusters["mean_delta_z"].abs())
        .groupby("feature_group", as_index=False)
        .agg(
            features=("feature", "count"),
            top_mentions=("profile_count", "sum"),
            mean_abs_delta_z=("abs_mean_delta_z", "mean"),
            max_stability_score=("stability_score", "max"),
        )
        .sort_values(["top_mentions", "mean_abs_delta_z"], ascending=[False, False])
        .reset_index(drop=True)
    )
else:
    group_summary = pd.DataFrame(columns=["feature_group", "features", "top_mentions", "mean_abs_delta_z", "max_stability_score"])

clusters.to_csv(CARD / "poor_window_clusters.csv", index=False)
feature_clusters.to_csv(CARD / "feature_delta_clusters.csv", index=False)
group_summary.to_csv(CARD / "feature_group_consensus.csv", index=False)
display(clusters.head(12))
display(feature_clusters.head(15))
display(group_summary)


,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C003,2024-01-02,2024-07-22,3,30,"base_plain_contract__wf_2023_2024,trend_plain_...",-53.159309,-62.852677,4.604990,12.970351,0.168584,83.159309,92.852677,underparticipation,reentry_participation_or_behavior_clone
1,C002,2022-09-02,2022-12-29,3,26,"base_plain_contract__wf_2022,trend_plain_contr...",-8.783553,-15.350762,9.859511,13.201932,0.188911,38.783553,45.350762,underparticipation,reentry_participation_or_behavior_clone
2,C001,2022-05-09,2022-08-10,3,4,"base_plain_contract__wf_2022,trend_plain_contr...",-5.006739,-5.862868,8.763512,9.155560,0.133953,35.006739,35.862868,underparticipation,reentry_participation_or_behavior_clone


,feature,profile_count,mean_delta_z,mean_abs_delta_z,direction_consistency,mean_spearman_to_excess,profiles,stability_score,feature_group
0,mom_63,3,2.729865,2.729865,1.0,-0.081839,"base_plain_contract__wf_2023_2024,trend_plain_...",3.784396,other
1,vix_percentile_252,6,-1.563805,1.563805,1.0,0.189373,"base_plain_contract__wf_2022,base_plain_contra...",3.043023,volatility_stress
2,realized_kurtosis_21,3,2.193179,2.193179,1.0,-0.001138,"base_plain_contract__wf_2022,trend_plain_contr...",3.040392,other
3,drawdown_63,3,1.950018,1.950018,1.0,-0.180918,"base_plain_contract__wf_2023_2024,trend_plain_...",2.703298,drawdown_path
4,days_since_high_63,3,-1.719128,1.719128,1.0,0.020136,"base_plain_contract__wf_2023_2024,trend_plain_...",2.383217,other
5,regime_risk,6,-1.080489,1.080489,1.0,0.187229,"base_plain_contract__wf_2022,base_plain_contra...",2.102534,other
6,trend_slope_21,3,1.467926,1.467926,1.0,0.002227,"base_plain_contract__wf_2023_2024,trend_plain_...",2.034978,trend_momentum
7,tracking_error_vol_21,3,1.329057,1.329057,1.0,-0.021866,"base_plain_contract__wf_2022,trend_plain_contr...",1.842464,soxx_relative
8,sharpe_proxy_21,3,1.319278,1.319278,1.0,-0.207864,"base_plain_contract__wf_2023_2024,trend_plain_...",1.828908,other
9,vix_zscore,6,-0.860380,0.860380,1.0,0.183673,"base_plain_contract__wf_2022,base_plain_contra...",1.674223,volatility_stress


,feature_group,features,top_mentions,mean_abs_delta_z,max_stability_score
0,volatility_stress,13,50,0.788868,3.043023
1,other,12,48,1.100949,3.784396
2,trend_momentum,9,45,0.976217,2.034978
3,drawdown_path,4,15,1.057609,2.703298
4,soxx_relative,4,12,0.788390,1.842464
5,reversal_reentry,2,10,0.374844,1.177031


In [5]:

walkforward_scores = candidate_summary.rename(
    columns={
        "full_gate_all": "full_gate",
        "gate_loss": "weighted_objective_loss",
    }
).copy()
walkforward_scores["profile"] = walkforward_scores["candidate"]
walkforward_scores["relative_return"] = walkforward_scores["min_relative_return"]
walkforward_scores["max_dd"] = walkforward_scores["max_max_dd"]
walkforward_scores["dominant_failure_type"] = ""
for idx, row in walkforward_scores.iterrows():
    rel_failed = float(row["min_relative_return"]) < TARGET_RELATIVE_RETURN
    dd_failed = float(row["max_max_dd"]) > TARGET_MAX_DD
    if rel_failed and dd_failed:
        failure = "joint_tradeoff"
    elif dd_failed:
        failure = "drawdown_control"
    elif rel_failed:
        failure = "underparticipation"
    else:
        failure = "pass"
    walkforward_scores.at[idx, "dominant_failure_type"] = failure

handoff = derive_training_objective_handoff(
    walkforward_scores=walkforward_scores,
    poor_window_clusters=clusters,
    feature_groups=group_summary,
    router_selection=None,
)
review = render_training_objective_handoff(handoff)
(CARD / "training_objective_handoff.md").write_text(review, encoding="utf-8")
walkforward_scores.to_csv(CARD / "walkforward_objective_scores.csv", index=False)
print(review)
display(walkforward_scores[["candidate", "full_gate", "weighted_objective_loss", "min_relative_return", "max_max_dd", "dominant_failure_type"]])


# Training Objective Handoff

## Decision

- decision: `train_state_conditioned_multi_objective`
- promotion: `blocked`
- selected_diagnostic_candidate: `trend_plain_contract`

## Objective Terms

- `walkforward_relative_return_gap_penalty`
- `walkforward_drawdown_gap_penalty`
- `high_vol_rebound_participation_floor`
- `downtrend_drawdown_state_cap`

## Risk Gap

- max_dd_gap=5.2pp, relative_gap=21.78pp, worst_window=``

## Primary Underparticipation Cluster

- cluster=`C003`, start=2024-01-02, end=2024-07-22, mean_excess=-53.16pp



,candidate,full_gate,weighted_objective_loss,min_relative_return,max_max_dd,dominant_failure_type
0,trend_plain_contract,False,73.765009,8.224563,20.198957,joint_tradeoff
1,base_plain_contract,False,142.004581,-0.136944,20.987807,joint_tradeoff
2,trend_plus_slope_low_cap15_weighted,False,163.113270,6.334730,28.944800,joint_tradeoff


In [6]:

selected_candidate = str(handoff.get("selected_diagnostic_candidate", ""))
selected_row = walkforward_scores[walkforward_scores["candidate"].eq(selected_candidate)].iloc[0].to_dict() if selected_candidate in set(walkforward_scores["candidate"]) else {}
cluster_types = clusters["dominant_failure_type"].value_counts().to_dict() if not clusters.empty else {}
selection = {
    "experiment": EXP,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": str(handoff.get("decision", "")),
    "role": "diagnose",
    "profile": PROFILE_NAME,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": [SOURCE_EXP],
    "closest_historical_success_case": "sequence_trend_joint_warm_start_official_eval_v1 established warm-start official daily paths as gate source",
    "closest_historical_failure_case": "sequence_trend_joint_warm_start_official_eval_v1 had zero full-gate candidates",
    "why_not_duplicate": "This notebook uses the failed warm-start official daily paths to locate poor windows and feature states, then routes a training objective instead of rerunning the same BC target screen.",
    "changed_layer": ["model_selection", "training_objective_handoff", "feature_delta_diagnostic"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "window_size": WINDOW_SIZE,
    "selected_diagnostic_candidate": selected_candidate,
    "selected_row": selected_row,
    "handoff": handoff,
    "poor_window_cluster_count": int(len(clusters)),
    "poor_window_cluster_types": cluster_types,
    "top_feature_groups": group_summary.head(5).to_dict("records"),
    "promotion_allowed": False,
    "training_allowed_from_this_card": False,
    "ppo_screen_allowed_next": False,
    "next_mechanism": str(handoff.get("decision", "")),
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": [
        "feature_cache_summary.csv",
        "diagnostic_manifest.csv",
        "diagnostics/*/poor_windows.csv",
        "diagnostics/*/feature_differences.csv",
        "poor_window_clusters.csv",
        "feature_delta_clusters.csv",
        "feature_group_consensus.csv",
        "walkforward_objective_scores.csv",
        "training_objective_handoff.md",
        "selection.json",
        "summary.json",
    ],
}
summary = {
    **selection,
    "candidate_summary": candidate_summary.to_dict("records"),
    "cluster_head": clusters.head(20).to_dict("records"),
    "feature_cluster_head": feature_clusters.head(30).to_dict("records"),
    "conclusion": [],
}
summary["conclusion"].append(
    f"Handoff decision is `{selection['decision']}` for `{selected_candidate}`; selected min relative "
    f"{float(selected_row.get('min_relative_return', 0.0)):.2f}pp and max MaxDD "
    f"{float(selected_row.get('max_max_dd', 0.0)):.2f}% remain outside the +30pp / <=15% full gate."
)
if not clusters.empty:
    top_cluster = clusters.iloc[0].to_dict()
    summary["conclusion"].append(
        f"Top poor-window cluster is `{top_cluster.get('dominant_failure_type')}` from {top_cluster.get('cluster_start')} to {top_cluster.get('cluster_end')}, "
        f"mean excess {float(top_cluster.get('mean_excess_return', 0.0)):.2f}pp, max MaxDD {float(top_cluster.get('max_max_dd', 0.0)):.2f}%."
    )
if not group_summary.empty:
    summary["conclusion"].append(
        "Top feature groups in bad windows: " + ", ".join(str(x) for x in group_summary["feature_group"].head(3).tolist()) + "."
    )
summary["conclusion"].append(
    "No promotion and no PPO handoff from this card; next work should design a state-conditioned multi-objective training target with both drawdown and participation terms."
)
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)
display(Markdown("\n".join(f"- {item}" for item in summary["conclusion"])))
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


- Handoff decision is `train_state_conditioned_multi_objective` for `trend_plain_contract`; selected min relative 8.22pp and max MaxDD 20.20% remain outside the +30pp / <=15% full gate.
- Top poor-window cluster is `underparticipation` from 2024-01-02 to 2024-07-22, mean excess -53.16pp, max MaxDD 12.97%.
- Top feature groups in bad windows: volatility_stress, other, trend_momentum.
- No promotion and no PPO handoff from this card; next work should design a state-conditioned multi-objective training target with both drawdown and participation terms.

{
  "experiment": "warm_start_official_failure_handoff_v1",
  "created_at": "2026-05-26T08:56:10.083969+00:00",
  "decision": "train_state_conditioned_multi_objective",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": [
    "sequence_trend_joint_warm_start_official_eval_v1"
  ],
  "closest_historical_success_case": "sequence_trend_joint_warm_start_official_eval_v1 established warm-start official daily paths as gate source",
  "closest_historical_failure_case": "sequence_trend_joint_warm_start_official_eval_v1 had zero full-gate candidates",
  "why_not_duplicate": "This notebook uses the failed warm-start official daily paths to locate poor windows and feature states, then routes a training objective instead of rerunning the same BC target screen.",
  "changed_layer": [
    "model_selection",
    "training_objective_handoff",
    "feature_delta_diagnostic"
  ],
  "target

## Conclusion

The final decision and single-run conclusion are written by the last code cell to `selection.json` and `summary.json`. This notebook is a handoff artifact: it chooses the next training-objective family from official warm-start failure evidence, but does not authorize runtime promotion or PPO.